# Simulación nula con la estructura real de VitalDB (punto D02 de la Dra. Gaur)

**Autor:** Aly Gallo · Cuaderno 2 de 2 (el 1 es *VitalDB_viabilidad_ChroniCare.ipynb*)

**Pregunta:** si usamos la prueba principal del protocolo (diferencia de pendientes entre el percentil 0,90 y el 0,50, Δβ) sobre una cohorte con la misma estructura que VitalDB, ¿cuántas veces "encuentra un techo" cuando no existe, y cuántas lo encuentra cuando sí existe?

**Cómo funciona:**
1. Toma la cohorte real de VitalDB (PCR, albúmina y leucocitos antes y ≥3 mediciones después, ventana W = 7 días) y usa **solo** sus valores previos y **los momentos reales en que se midió** después.
2. Con eso calcula un índice de estado biológico (BSI) previo: promedio de tres puntajes estandarizados, sin entrenar con ningún desenlace.
3. **Inventa** la recuperación en dos mundos: *sin techo* y *con techo* (el estado biológico limita cuánto se puede recuperar).
4. Mide la recuperación como lo haría el protocolo (fracción de la caída recuperada, con nadir robusto) y aplica la prueba.

**No usa ningún desenlace real de salud.** La recuperación es simulada. El BSI usa PCR, albúmina y leucocitos, igual que la recuperación observada, y eso crea un acoplamiento matemático que la simulación mide a propósito.

**Valores fijados de antemano:** semilla 2026; BSI = promedio de z(−log PCR), z(albúmina), z(−log leucocitos); recuperación = (valor final − nadir)/(línea base − nadir).

In [ ]:
import sys, datetime, json, time, warnings
warnings.filterwarnings("ignore")   # silencia avisos de convergencia de la regresion cuantilica
import numpy as np, pandas as pd
from statsmodels.regression.quantile_regression import QuantReg
print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

W = 7.0                       # ventana de recuperacion (dias)
PRE_DIAS = 7                  # medicion previa: hasta 7 dias antes
MIN_POST = 3                  # minimo de mediciones dentro de W
SEMILLA = 2026
N_REPS = 300                  # repeticiones por escenario
TAMANOS = [300, 500, None]    # None = cohorte completa
RUIDOS = [0.3, 0.6]           # ruido de medicion (en desviaciones estandar del BSI)
ACOPLAMIENTOS = [0.0, 0.3]    # cuanto depende la profundidad de la caida del estado previo
TECHOS = [0.3, 0.5, 1.0]      # fuerza del techo: debil, moderado, fuerte
D0, T0, TAU = 1.0, 2.0, 2.0   # caida media (SD), dia del nadir, constante de recuperacion (dias)
UMBRAL_CAIDA = 0.5            # solo se analizan pacientes con caida observada > 0.5 SD


## Paso 1. Datos y cohorte real

In [ ]:
lab = pd.read_csv("https://api.vitaldb.net/labs")
cases = pd.read_csv("https://api.vitaldb.net/cases")
print("lab:", lab.shape, "| cases:", cases.shape)
d = lab.merge(cases[["caseid", "subjectid", "opend"]], on="caseid", how="left")
d["dias_post"] = (d.dt - d.opend) / 86400
d["dias_pre"] = -d.dt / 86400

pruebas = ["crp", "alb", "wbc"]
conjuntos = []
for t in pruebas:
    x = d[d.name == t]
    pre = set(x[(x.dt <= 0) & (x.dias_pre <= PRE_DIAS)].caseid)
    p = x[(x.dias_post >= 0) & (x.dias_post <= W)].groupby("caseid").dias_post.agg(["size", "max"])
    post = set(p[(p["size"] >= MIN_POST) & (p["max"] >= W / 2)].index)
    conjuntos.append(pre & post)
coh = set.intersection(*conjuntos)
print("Cirugias en la cohorte (W = 7):", len(coh), "(debe coincidir con el cuaderno 1: 774)")

# una cirugia por paciente (la primera) para no repetir personas
c = cases[cases.caseid.isin(coh)].sort_values("caseid").drop_duplicates("subjectid", keep="first")
ids = c.caseid.tolist()
print("Pacientes unicos en la cohorte:", len(ids))


## Paso 2. BSI previo real y momentos reales de medicion

In [ ]:
base = {}
for t in pruebas:
    x = d[(d.name == t) & d.caseid.isin(ids) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)]
    base[t] = x.sort_values("dt").groupby("caseid").result.last()
B = pd.DataFrame(base).loc[ids]
z = lambda s: (s - s.mean()) / s.std()
bsi_real = pd.concat([z(-np.log(B.crp + 0.1)), z(B.alb), z(-np.log(B.wbc))], axis=1).mean(axis=1)
bsi_real = z(bsi_real).values            # estado previo "verdadero" para la simulacion (SD = 1)

# momentos reales de medicion despues de la cirugia (se usan los de la PCR)
xc = d[(d.name == "crp") & d.caseid.isin(ids) & (d.dias_post >= 0) & (d.dias_post <= W)]
tiempos = [np.sort(xc[xc.caseid == i].dias_post.values) for i in ids]
m = max(len(t) for t in tiempos)
T = np.full((len(ids), m), np.nan)
for k, t in enumerate(tiempos):
    T[k, :len(t)] = t
print("Mediciones por paciente: mediana", int(np.median([len(t) for t in tiempos])), "| maximo", m)
print("Primer dia medido, mediana:", round(np.nanmedian(np.nanmin(T, axis=1)), 2))


## Paso 3. El simulador
Para cada paciente: se inventa una recuperación verdadera `r` (fracción de la caída que se recupera al final de la ventana), se genera la trayectoria del BSI con un nadir al día 2 y se "mide" con ruido **solo en los momentos reales**. Luego se estima la recuperación como lo haría el protocolo y se aplica la prueba Δβ = β(0,90) − β(0,50).

In [ ]:
def simular(rng, idx, ruido, acople, techo):
    n = len(idx)
    Tn = T[idx]
    mask = ~np.isnan(Tn)
    Bt = bsi_real[idx]                                   # estado previo verdadero
    D = np.clip(D0 * (1 + acople * Bt), 0.3, None)       # profundidad de la caida
    u = rng.beta(4, 2, n)                                # recuperacion potencial
    r = u if techo is None else np.minimum(u, np.clip(0.95 + techo * Bt, 0.05, 1.0))
    # trayectoria: baja hasta T0 y luego recupera una fraccion r al dia W
    k = 1 - np.exp(-(W - T0) / TAU)
    t = np.where(mask, Tn, 0.0)
    rec = r[:, None] * (1 - np.exp(-(t - T0) / TAU)) / k
    q = np.where(t <= T0, t / T0, 1 - rec)
    x = Bt[:, None] - D[:, None] * q
    obs = x + rng.normal(0, ruido, x.shape)
    obs = np.where(mask, obs, np.nan)
    base_obs = Bt + rng.normal(0, ruido, n)              # la linea base tambien se mide con ruido
    # estimacion estilo protocolo: nadir robusto = media de los 2 valores mas bajos; final = ultima medicion
    srt = np.sort(np.where(mask, obs, np.inf), axis=1)
    nadir = srt[:, :2].mean(axis=1)
    last = np.nanargmax(np.where(mask, Tn, -np.inf), axis=1)
    fin = obs[np.arange(n), last]
    caida = base_obs - nadir
    ok = caida > UMBRAL_CAIDA
    R = np.clip((fin - nadir) / np.where(ok, caida, 1), -0.5, 1.5)
    S = base_obs
    return S[ok], R[ok]

def delta_beta(S, R):
    X = np.column_stack([np.ones(len(S)), S])
    b90 = QuantReg(R, X).fit(q=0.90, max_iter=2000).params[1]
    b50 = QuantReg(R, X).fit(q=0.50, max_iter=2000).params[1]
    return b90 - b50, b50, b90


## Paso 4. Correr los escenarios
Tarda unos minutos. Por escenario: Δβ bajo *sin techo* (nulo), valor crítico (percentil 95 del nulo) y poder bajo *con techo*.

In [ ]:
rng = np.random.default_rng(SEMILLA)
N = len(ids)
filas = []
t_ini = time.time()
for n_obj in TAMANOS:
    n_usar = N if n_obj is None else min(n_obj, N)
    for ruido in RUIDOS:
        for acople in ACOPLAMIENTOS:
            nulo = []
            for _ in range(N_REPS):
                idx = rng.choice(N, n_usar, replace=False)
                S, R = simular(rng, idx, ruido, acople, None)
                nulo.append(delta_beta(S, R)[0])
            nulo = np.array(nulo)
            crit = np.percentile(nulo, 95)
            fila = {"n": n_usar, "ruido": ruido, "acople": acople,
                    "Δβ nulo, media": round(nulo.mean(), 3), "Δβ nulo, p95 (critico)": round(crit, 3)}
            for tk in TECHOS:
                alt = []
                for _ in range(N_REPS):
                    idx = rng.choice(N, n_usar, replace=False)
                    S, R = simular(rng, idx, ruido, acople, tk)
                    alt.append(delta_beta(S, R)[0])
                fila[f"poder techo {tk}"] = round(float(np.mean(np.array(alt) > crit)), 2)
            filas.append(fila)
    print("n =", n_usar, "listo;", round(time.time() - t_ini), "s")
resultados = pd.DataFrame(filas)
print(resultados.to_string(index=False))


## Paso 4b. Control de sesgo con una muestra enorme (n = 20 000)
Si aun con 20 000 pacientes simulados la prueba no distingue el mundo con techo del mundo sin techo, o si el mundo *sin techo* ya da un Δβ distinto de 0, el problema es del **método** (ruido y acoplamiento) y no del tamaño de muestra.

In [ ]:
rng2 = np.random.default_rng(SEMILLA + 1)
M_GRANDE = 20000
sel = rng2.integers(0, N, M_GRANDE)          # remuestreo de los pacientes reales
T_real, bsi_real_orig = T.copy(), bsi_real.copy()
T, bsi_real = T_real[sel], bsi_real_orig[sel]
filas_c = []
for ruido in [0.1, 0.3, 0.6]:
    for acople in ACOPLAMIENTOS:
        fila = {"ruido": ruido, "acople": acople}
        for nombre, tk in [("sin techo", None), ("techo 0.5", 0.5), ("techo 1.0", 1.0)]:
            S, R = simular(rng2, np.arange(M_GRANDE), ruido, acople, tk)
            fila["Δβ " + nombre] = round(delta_beta(S, R)[0], 3)
        filas_c.append(fila)
T, bsi_real = T_real, bsi_real_orig          # se restauran los datos reales
control = pd.DataFrame(filas_c)
print(control.to_string(index=False))


## Paso 5. Guardar

In [ ]:
resultados.to_csv("simulacion_nula_vitaldb.csv", index=False)
control.to_csv("simulacion_nula_control_n20000.csv", index=False)
meta = {"fecha": str(datetime.date.today()), "semilla": SEMILLA, "n_reps": N_REPS, "cohorte_n": int(N),
        "W": W, "pre_dias": PRE_DIAS, "min_post": MIN_POST, "D0": D0, "T0": T0, "TAU": TAU,
        "umbral_caida": UMBRAL_CAIDA, "ruidos": RUIDOS, "acoplamientos": ACOPLAMIENTOS, "techos": TECHOS}
with open("simulacion_nula_parametros.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Guardado: simulacion_nula_vitaldb.csv y simulacion_nula_parametros.json")


## Cómo leer la tabla
- **Δβ nulo, media:** si es distinto de 0 en el mundo *sin techo*, la prueba fabrica un techo falso (sesgo por acoplamiento o por ruido). Cuanto más lejos de 0, peor.
- **Δβ nulo, p95 (crítico):** el valor que Δβ supera solo el 5 % de las veces cuando no hay techo. Es el umbral calibrado que se usaría en el análisis real.
- **poder techo 0.3 / 0.5 / 1.0:** con ese umbral, la fracción de veces que la prueba detecta un techo que sí existe (débil, moderado, fuerte). Se busca ≥ 0,80.
- **Control con n = 20 000:** muestra el Δβ en cada mundo sin la limitación del tamaño. Si el mundo *sin techo* da Δβ negativo o lejos de 0, hay un sesgo del método (ruido de medición y acoplamiento) que no desaparece con más pacientes.
- Se resume en una línea por tamaño de muestra: ¿desde qué n el poder llega a 0,80 con ruido y acoplamiento realistas?

**Limites:** la forma de la recuperación y del techo es una suposición; las conclusiones valen para esa suposición. Con datos reales de INSPIRE se repetiría con sus propios tiempos.